<a href="https://colab.research.google.com/github/Solmaeir/Roman_Columns/blob/main/Roman_Columns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roma Dönemi Kolon Restorasyonu
**Görüntü İşleme Projesi**

Pipeline:
1. Görüntü ön işleme (Bilateral Filter + CLAHE)
2. Kolon segmentasyonu (SAM → Klasik CV fallback)
3. İdeal kolon formu hesaplama
4. Simetri tabanlı restorasyon
5. İnpainting ile pürüzsüzleştirme (LaMa → OpenCV TELEA fallback)

In [ ]:
# ─── Google Drive bağla ve klasör yollarını tanımla ───
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    BASE          = '/content/drive/MyDrive/sutun_veriseti_1'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Google Colab — Drive bağlandı.')
except ImportError:
    IN_COLAB = False
    BASE          = '.'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Yerel ortam.')

damaged_files   = [f for f in os.listdir(DAMAGED_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
reference_files = [f for f in os.listdir(REFERENCE_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f'Damaged  : {len(damaged_files)} görüntü')
print(f'Reference: {len(reference_files)} görüntü')
print(f'Test [25]: {damaged_files[25]}')

In [ ]:
# ─── Bağımlılıkları kur (sürüm çakışması YOK) ───
# NOT: pillow==9.5.0 ZORLA KURULMAZ — scikit-image ile çakışır.
# SAM zaten yeni Pillow ile çalışır.

import subprocess, sys

def pip_install(pkg, quiet=True):
    args = [sys.executable, '-m', 'pip', 'install', pkg]
    if quiet:
        args.append('-q')
    subprocess.run(args, check=False)

# SAM
try:
    import segment_anything
    print('segment-anything zaten kurulu.')
except ImportError:
    print('segment-anything kuruluyor...')
    pip_install('segment-anything')

# LaMa (opsiyonel)
try:
    import simple_lama_inpainting
    print('simple-lama-inpainting zaten kurulu.')
except ImportError:
    print('simple-lama-inpainting kuruluyor...')
    pip_install('simple-lama-inpainting')

# SAM checkpoint
CHECKPOINT = 'sam_vit_b_01ec64.pth'
if not os.path.exists(CHECKPOINT):
    print('SAM checkpoint indiriliyor...')
    subprocess.run(['wget', '-q',
                    'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'],
                   check=False)
    print('Checkpoint indirildi.')
else:
    print('SAM checkpoint mevcut.')

In [ ]:
# ─── Import'lar (graceful fallback ile) ───
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'Torch  : {torch.__version__}')
print(f'PIL    : {Image.__version__}')

# SAM
SAM_AVAILABLE = False
try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    SAM_AVAILABLE = True
    print(f'SAM    : OK  (CUDA: {torch.cuda.is_available()})')
except Exception as e:
    print(f'SAM    : Yok ({e}) → Klasik CV kullanılacak')

# LaMa
LAMA_AVAILABLE = False
try:
    from simple_lama_inpainting import SimpleLama
    LAMA_AVAILABLE = True
    print('LaMa   : OK')
except Exception as e:
    print(f'LaMa   : Yok ({e}) → OpenCV TELEA kullanılacak')

In [ ]:
# ─── AI modellerini yükle ───
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# SAM
mask_generator = None
if SAM_AVAILABLE:
    try:
        _sam = sam_model_registry['vit_b'](checkpoint=CHECKPOINT)
        _sam.to(DEVICE)
        mask_generator = SamAutomaticMaskGenerator(
            model=_sam,
            points_per_side=32,
            pred_iou_thresh=0.86,
            stability_score_thresh=0.90,
            min_mask_region_area=300,
        )
        print(f'SAM modeli yüklendi ({DEVICE}).')
    except Exception as e:
        print(f'SAM yüklenemedi: {e}')
        SAM_AVAILABLE = False

# LaMa
lama_model = None
if LAMA_AVAILABLE:
    try:
        lama_model = SimpleLama()
        print('LaMa modeli yüklendi.')
    except Exception as e:
        print(f'LaMa yüklenemedi: {e}')
        LAMA_AVAILABLE = False

if not SAM_AVAILABLE:
    print('→ Klasik CV (Canny + kontur) ile devam edilecek.')
if not LAMA_AVAILABLE:
    print('→ OpenCV TELEA inpainting ile devam edilecek.')

In [ ]:
# ─── Görüntü ön işleme ───

def preprocess(img_bgr):
    """Bilateral filtre + CLAHE ile kontrast iyileştir."""
    denoised = cv2.bilateralFilter(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

print('preprocess() hazır.')

In [ ]:
# ─── Kolon segmentasyonu: SAM + Klasik CV fallback ───

def _score_contour(cnt, h_img, w_img):
    """Konturun 'kolon olma' skorunu hesapla."""
    area = cv2.contourArea(cnt)
    if area < 500:
        return 0.0
    x, y, w, h = cv2.boundingRect(cnt)
    aspect = h / (w + 1e-5)
    area_ratio = area / (h_img * w_img)
    if aspect > 1.5 and 0.03 < area_ratio < 0.75:
        return aspect * area_ratio
    return 0.0


def detect_column_classical(img_bgr):
    """
    Klasik CV tabanlı kolon maskesi:
    Canny kenar → morfolojik kapama → kontur seçimi.
    """
    h_img, w_img = img_bgr.shape[:2]
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptif eşikleme + Canny
    adapt = cv2.adaptiveThreshold(blurred, 255,
                                  cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY_INV, 31, 5)
    edges = cv2.Canny(blurred, 20, 80)
    combined = cv2.bitwise_or(adapt, edges)

    kern_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    kern_ellp = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    closed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kern_rect, iterations=4)
    dilated = cv2.dilate(closed, kern_rect, iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    scored = [(cnt, _score_contour(cnt, h_img, w_img)) for cnt in contours]
    scored.sort(key=lambda x: x[1], reverse=True)

    # En iyi skoru olan; skor 0 ise en büyük konturu al
    best_cnt = scored[0][0] if scored[0][1] > 0 else max(contours, key=cv2.contourArea)

    mask = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.drawContours(mask, [best_cnt], -1, 255, -1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern_ellp, iterations=5)
    return mask


def detect_column_sam(img_bgr):
    """SAM tabanlı kolon maskesi."""
    if mask_generator is None:
        return None
    h_img, w_img = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    try:
        masks = mask_generator.generate(img_rgb)
    except Exception as e:
        print(f'  SAM hatası: {e}')
        return None

    if not masks:
        return None

    masks_sorted = sorted(masks, key=lambda m: m['area'], reverse=True)
    best_mask, best_score = None, 0.0

    for m in masks_sorted[:12]:
        seg = m['segmentation'].astype(np.uint8) * 255
        cnts, _ = cv2.findContours(seg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        cnt = max(cnts, key=cv2.contourArea)
        score = _score_contour(cnt, h_img, w_img)
        if score > best_score:
            best_score = score
            best_mask  = seg

    if best_mask is None:
        best_mask = masks_sorted[0]['segmentation'].astype(np.uint8) * 255

    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    best_mask = cv2.morphologyEx(best_mask, cv2.MORPH_CLOSE, kern, iterations=4)
    return best_mask


def detect_column(img_bgr):
    """SAM varsa SAM, yoksa klasik CV."""
    if SAM_AVAILABLE:
        mask = detect_column_sam(img_bgr)
        if mask is not None:
            print('  Kolon SAM ile tespit edildi.')
            return mask, 'SAM'
    mask = detect_column_classical(img_bgr)
    if mask is not None:
        print('  Kolon Klasik CV ile tespit edildi.')
        return mask, 'Klasik CV'
    return None, None

print('Segmentasyon fonksiyonları hazır.')

In [ ]:
# ─── Simetri tabanlı restorasyon ───

def get_column_bbox(column_mask):
    """Kolon maskesinin bounding box ve simetri eksenini döndür."""
    cnts, _ = cv2.findContours(column_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(cnt)
    cx = x + w // 2
    return x, y, w, h, cx, cnt


def create_ideal_mask(column_mask, expand_top=0.20, expand_bottom=0.05):
    """
    Mevcut kolon maskesinin bounding box'ını biraz genişleterek
    'ideal' (tam, hasarsız) kolon alanını tanımla.
    """
    h_img, w_img = column_mask.shape[:2]
    info = get_column_bbox(column_mask)
    if info is None:
        return None
    x, y, w, h, cx, _ = info

    pad_top = int(h * expand_top)
    pad_bot = int(h * expand_bottom)

    ideal = np.zeros((h_img, w_img), dtype=np.uint8)
    y1 = max(0, y - pad_top)
    y2 = min(h_img, y + h + pad_bot)
    ideal[y1:y2, x:x + w] = 255
    return ideal


def symmetry_restore(img_bgr, column_mask):
    """
    Simetri tabanlı restorasyon:
    1. Simetri eksenini bul
    2. Her iki yarıdaki piksel sayısını karşılaştır
    3. Sağlam tarafı aynalayarak hasarlı tarafa uygula
    4. Geçişi yumuşatmak için Poisson klonlama (seamlessClone)
    """
    h_img, w_img = img_bgr.shape[:2]
    info = get_column_bbox(column_mask)
    if info is None:
        return img_bgr.copy(), None

    x, y, w, h, cx, main_cnt = info

    left_px  = np.sum(column_mask[:, x:cx]    > 128)
    right_px = np.sum(column_mask[:, cx:x + w] > 128)

    result = img_bgr.copy()

    if left_px >= right_px:
        # Sol daha sağlam → sağa aynala
        good   = img_bgr[:, x:cx].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xs, xe = cx, min(w_img, cx + mw)
        aw     = xe - xs
        roi_mask = (column_mask[:, xs:xe] == 0)  # sadece boş alanlara yaz
        tmp = result[:, xs:xe].copy()
        tmp[roi_mask] = mirror[:, :aw][roi_mask]
        result[:, xs:xe] = tmp
        direction = 'sağ taraf tamamlandı (sol referans)'
    else:
        # Sağ daha sağlam → sola aynala
        good   = img_bgr[:, cx:x + w].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xe, xs = cx, max(0, cx - mw)
        aw     = xe - xs
        roi_mask = (column_mask[:, xs:xe] == 0)
        tmp = result[:, xs:xe].copy()
        tmp[roi_mask] = mirror[:, mw - aw:][roi_mask]
        result[:, xs:xe] = tmp
        direction = 'sol taraf tamamlandı (sağ referans)'

    print(f'  Simetri: {direction} | Sol={left_px}px, Sağ={right_px}px')
    return result, cx


print('Simetri fonksiyonları hazır.')

In [ ]:
# ─── İnpainting: LaMa → OpenCV TELEA fallback ───

def inpaint(img_bgr, mask_u8):
    """
    mask_u8: 0/255 maske — 255 olan bölgeler doldurulur.
    LaMa varsa LaMa, yoksa OpenCV TELEA kullanır.
    """
    if np.sum(mask_u8) < 100:
        return img_bgr.copy()

    if LAMA_AVAILABLE and lama_model is not None:
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_pil  = Image.fromarray(img_rgb)
        mask_pil = Image.fromarray(mask_u8)
        try:
            result_pil = lama_model(img_pil, mask_pil)
            result_rgb = np.array(result_pil)
            return cv2.cvtColor(result_rgb, cv2.COLOR_RGB2BGR)
        except Exception as e:
            print(f'  LaMa hatası: {e} → OpenCV TELEA kullanılıyor')

    # OpenCV TELEA (her zaman çalışır)
    return cv2.inpaint(img_bgr, mask_u8, inpaintRadius=7, flags=cv2.INPAINT_TELEA)


print('inpaint() hazır.')

In [ ]:
# ─── Ana Restorasyon Pipeline ───

def restore_column(img_path, expand_top=0.20, expand_bottom=0.05):
    """
    Hasarlı kolon görüntüsünü restore eder.

    Adımlar:
    1. Yükle + ön işle
    2. Kolon segmentasyonu
    3. İdeal form + eksik bölge maskesi
    4. Simetri restorasyon
    5. İnpainting (simetri sınırlarını yumuşat)
    6. Son filtre + görselleştir
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'Görüntü okunamadı: {img_path}')

    fname = os.path.basename(img_path)
    print(f"{'='*65}")
    print(f'  Görüntü : {fname}')
    print(f'  Boyut   : {img.shape[1]} x {img.shape[0]} px')
    print(f"{'='*65}")

    # ── 1. Ön işleme ──
    print('Adım 1 : Bilateral + CLAHE...')
    proc = preprocess(img)

    # ── 2. Segmentasyon ──
    print('Adım 2 : Kolon segmentasyonu...')
    col_mask, method = detect_column(proc)
    if col_mask is None:
        print('HATA: Kolon tespit edilemedi.')
        return None

    # ── 3. İdeal form + eksik bölge ──
    print('Adım 3 : İdeal form hesaplanıyor...')
    ideal_mask = create_ideal_mask(col_mask, expand_top, expand_bottom)
    if ideal_mask is None:
        print('HATA: İdeal form oluşturulamadı.')
        return None

    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    col_filled   = cv2.morphologyEx(col_mask, cv2.MORPH_CLOSE, kern, iterations=3)
    missing_mask = cv2.bitwise_and(ideal_mask, cv2.bitwise_not(col_filled))
    missing_mask = cv2.morphologyEx(missing_mask, cv2.MORPH_OPEN, kern, iterations=1)
    print(f'  Eksik piksel: {np.sum(missing_mask > 0):,}')

    # ── 4. Simetri restorasyon ──
    print('Adım 4 : Simetri restorasyon...')
    sym_bgr, sym_axis = symmetry_restore(proc, col_mask)

    # ── 5. İnpainting ──
    print('Adım 5 : İnpainting...')
    if np.sum(missing_mask) > 200:
        final_bgr = inpaint(sym_bgr, missing_mask)
        inpaint_method = 'LaMa' if LAMA_AVAILABLE else 'OpenCV TELEA'
        print(f'  İnpainting yöntemi: {inpaint_method}')
    else:
        final_bgr = sym_bgr
        print('  (Eksik alan çok küçük, inpainting atlandı)')

    # ── 6. Son post-process ──
    final_bgr = cv2.bilateralFilter(final_bgr, d=7, sigmaColor=50, sigmaSpace=50)

    # ── RGB'ye çevir ──
    img_rgb   = cv2.cvtColor(img,     cv2.COLOR_BGR2RGB)
    proc_rgb  = cv2.cvtColor(proc,    cv2.COLOR_BGR2RGB)
    sym_rgb   = cv2.cvtColor(sym_bgr,  cv2.COLOR_BGR2RGB)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    # ── Simetri ekseni görseli ──
    vis_sym = img_rgb.copy()
    if sym_axis is not None:
        cv2.line(vis_sym, (sym_axis, 0), (sym_axis, img.shape[0]), (255, 50, 50), 2)
    cnts, _ = cv2.findContours(col_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        cv2.drawContours(vis_sym, [max(cnts, key=cv2.contourArea)], -1, (50, 255, 50), 2)

    # ── Görselleştirme ──
    fig, axes = plt.subplots(1, 6, figsize=(30, 8))

    axes[0].imshow(img_rgb)
    axes[0].set_title('① Orijinal\n(Hasarlı)', fontsize=12, fontweight='bold', color='darkred')

    axes[1].imshow(proc_rgb)
    axes[1].set_title('② Ön İşlenmiş\nBilateral + CLAHE', fontsize=11)

    axes[2].imshow(col_mask, cmap='gray')
    axes[2].set_title(f'③ Kolon Maskesi\n({method})', fontsize=11)

    axes[3].imshow(vis_sym)
    axes[3].set_title('④ Simetri Analizi\nKırmızı=Eksen | Yeşil=Kolon', fontsize=11)

    axes[4].imshow(missing_mask, cmap='hot')
    axes[4].set_title('⑤ Eksik Bölge\n(Dolacak Alan)', fontsize=11)

    axes[5].imshow(final_rgb)
    axes[5].set_title('⑥ RESTORE EDİLMİŞ\nSimetri + İnpaint', fontsize=12,
                      fontweight='bold', color='darkgreen')

    for ax in axes:
        ax.axis('off')

    plt.suptitle(f'Roma Kolon Restorasyonu — {fname}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('restoration_steps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Adım görüntüsü 'restoration_steps.png' olarak kaydedildi.")

    return {
        'original':     img_rgb,
        'preprocessed': proc_rgb,
        'column_mask':  col_mask,
        'missing_mask': missing_mask,
        'symmetry':     sym_rgb,
        'final':        final_rgb,
        'method':       method,
        'sym_axis':     sym_axis,
    }

print('restore_column() hazır.')

In [ ]:
# ─── Test: damaged_files[25] ───
# a9ca5f5b6ba9cd36e161e86f0fa4925b.jpg

damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

TEST_IDX  = 25
test_file = damaged_files[TEST_IDX]
test_path = os.path.join(DAMAGED_DIR, test_file)

print(f'Test görüntüsü [{TEST_IDX}]: {test_file}')
print(f'Yol: {test_path}')

result = restore_column(test_path, expand_top=0.20, expand_bottom=0.05)

In [ ]:
# ─── Önce / Sonra karşılaştırması ───

if result is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 12))

    ax1.imshow(result['original'])
    ax1.set_title('ÖNCE\n(Hasarlı Kolon)', fontsize=16, fontweight='bold', color='darkred')
    ax1.axis('off')

    ax2.imshow(result['final'])
    ax2.set_title('SONRA\n(Restore Edilmiş)', fontsize=16, fontweight='bold', color='darkgreen')
    ax2.axis('off')

    plt.suptitle('Roma Kolon Dijital Restorasyonu', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Karşılaştırma 'before_after.png' olarak kaydedildi.")
else:
    print('Pipeline çalışmadı. Hata mesajlarını kontrol et.')

In [ ]:
# ─── (OPSİYONEL) Toplu işleme: Seçili iyi görüntüler ───
# Bu hücreyi çalıştırmak zorunlu değil.

GOOD_INDICES = [2, 3, 6, 9, 12, 14, 15, 17, 19, 20, 21, 25, 26, 27, 29]
SAVE_DIR     = '/content/drive/MyDrive/sutun_veriseti_1/restored' if IN_COLAB else './restored'
os.makedirs(SAVE_DIR, exist_ok=True)

damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for idx in GOOD_INDICES:
    if idx >= len(damaged_files):
        continue
    path = os.path.join(DAMAGED_DIR, damaged_files[idx])
    try:
        res = restore_column(path, expand_top=0.20, expand_bottom=0.05)
        if res is not None:
            save_path = os.path.join(SAVE_DIR, f'restored_{idx:02d}_{damaged_files[idx]}')
            final_bgr = cv2.cvtColor(res['final'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(save_path, final_bgr)
            print(f'  Kaydedildi: {save_path}')
    except Exception as e:
        print(f'  [{idx}] Hata: {e}')

print('Toplu işleme tamamlandı.')